In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import clear_output

PROJECT_ROOT = Path("../..").resolve()
MODEL_PATH = PROJECT_ROOT / "experiments" / "06_cross_view_mobilenetv2" / "best_mobilenetv2_crossview_finetuned.keras"
IMG_SIZE = (224, 224)
LABEL_NAMES = [
    "safe_driving", "texting_right", "phone_right", "texting_left", "phone_left",
    "adjusting_radio", "drinking", "reaching_behind", "hair_or_makeup", "talking_to_passenger",
]

VIDEO_PATH = r"D:\MSC_PROJECT\test_drive.mp4"  # change to your video's path
SAMPLE_EVERY_N_SECONDS = 0.5  # laptop GPU is fast enough to sample much more densely than the board
CONFIDENCE_THRESHOLD = 0.40
SHOW_EVERY_FRAME = True  # set False to just process silently and only print the final summary

model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded:", MODEL_PATH)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video at {VIDEO_PATH} - check the path")

video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frame_step = max(1, int(round(video_fps * SAMPLE_EVERY_N_SECONDS)))
print(f"Video FPS: {video_fps:.1f}, sampling every {frame_step} frames (~{SAMPLE_EVERY_N_SECONDS}s of footage)")

detections = []  # (video_time_s, class_name, confidence)
tick = 0
frame_index = 0

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("End of video reached.")
            break
        frame_index += 1
        if (frame_index - 1) % frame_step != 0:
            continue

        video_time_s = frame_index / video_fps

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = tf.image.resize(frame_rgb, IMG_SIZE)
        batch = tf.expand_dims(tf.cast(resized, tf.float32), axis=0)
        probs = model.predict(batch, verbose=0)[0]

        class_id = int(np.argmax(probs))
        confidence = float(probs[class_id])
        label = LABEL_NAMES[class_id]
        tick += 1

        is_distraction = class_id != 0 and confidence >= CONFIDENCE_THRESHOLD
        if is_distraction:
            detections.append((video_time_s, label, confidence))

        if SHOW_EVERY_FRAME:
            clear_output(wait=True)
            plt.figure(figsize=(6, 4.5))
            plt.imshow(frame_rgb)
            title_color = "red" if is_distraction else "black"
            plt.title(f"t={video_time_s:.1f}s | {label} ({confidence:.0%})", fontsize=11, color=title_color)
            plt.axis("off")
            plt.show()

except KeyboardInterrupt:
    print("Stopped by user.")

cap.release()
print(f"\nProcessed {tick} sampled frame(s).")

In [ ]:
print(f"Confident distraction detections: {len(detections)}")
for t, label, conf in detections:
    print(f"  t={t:.1f}s: {label} ({conf:.0%})")